In [42]:
import zipfile
import os

# Find any .zip file in /content/
zip_files = [f for f in os.listdir('/content/') if f.endswith('.zip')]

if not zip_files:
    print("No ZIP file found. Upload it first using the Files sidebar.")
else:
    zip_path = os.path.join('/content/', zip_files[0])
    print(f"Found: {zip_files[0]}")
    zipfile.ZipFile(zip_path, 'r').extractall('/content/mushroom_dataset')
    print("Extracted to /content/mushroom_dataset/")


Found: Mushrooms.yolov8.zip
Extracted to /content/mushroom_dataset/


In [43]:
#fix paths
yaml_path = '/content/mushroom_dataset/data.yaml'
with open(yaml_path) as f: lines = f.readlines()
with open(yaml_path, 'w') as f:
    for line in lines:
        if line.startswith('train:'): f.write('train: /content/mushroom_dataset/train/images\n')
        elif line.startswith('val:'): f.write('val: /content/mushroom_dataset/valid/images\n')
        elif line.startswith('test:'): f.write('test: /content/mushroom_dataset/test/images\n')
        else: f.write(line)


In [44]:
#install ultralytics
!pip install -q ultralytics
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — enable it in Runtime settings")


GPU: Tesla T4


In [45]:
#fix test and validation set split

import os
import shutil
import random

random.seed(42)

base = '/content/mushroom_dataset'
img_dir = os.path.join(base, 'train', 'images')
lbl_dir = os.path.join(base, 'train', 'labels')

imgs = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
random.shuffle(imgs)

n = len(imgs)
n_val = max(1, int(n * 0.10))
n_test = max(1, int(n * 0.10))

splits = {
    'valid': imgs[:n_val],
    'test': imgs[n_val:n_val + n_test],
    'train': imgs[n_val + n_test:]
}

for split_name, split_imgs in splits.items():
    os.makedirs(os.path.join(base, split_name, 'images'), exist_ok=True)
    os.makedirs(os.path.join(base, split_name, 'labels'), exist_ok=True)
    for img_name in split_imgs:
        stem = os.path.splitext(img_name)[0]
        for ext in ['.jpg', '.jpeg', '.png']:
            src_img = os.path.join(img_dir, stem + ext)
            if os.path.exists(src_img):
                shutil.move(src_img, os.path.join(base, split_name, 'images', stem + ext))
                break
        for ext in ['.txt']:
            src_lbl = os.path.join(lbl_dir, stem + ext)
            if os.path.exists(src_lbl):
                shutil.move(src_lbl, os.path.join(base, split_name, 'labels', stem + ext))

with open(os.path.join(base, 'data.yaml'), 'w') as f:
    f.write(f"train: {base}/train/images\n")
    f.write(f"val: {base}/valid/images\n")
    f.write(f"test: {base}/test/images\n")
    f.write("nc: 4\n")
    f.write("names: ['cap', 'coral', 'stem', 'underside']\n")

print(f"Train: {len(splits['train'])} | Val: {len(splits['valid'])} | Test: {len(splits['test'])}")


Train: 292 | Val: 36 | Test: 36


In [46]:
import os, glob
# 1. Check what datasets exist
!ls -R /content/mushroom_dataset/data.yaml
# 2. Verify class count in the yaml
!cat /content/mushroom_dataset/data.yaml
# 3. Check for old run folders
!ls -la /content/runs/segment/
# 4. If in doubt, wipe and re-extract
!rm -rf /content/mushroom_dataset /content/runs
!unzip -q /content/Mushrooms.yolov8.zip -d /content/mushroom_dataset


/content/mushroom_dataset/data.yaml
train: /content/mushroom_dataset/train/images
val: /content/mushroom_dataset/valid/images
test: /content/mushroom_dataset/test/images
nc: 4
names: ['cap', 'coral', 'stem', 'underside']
ls: cannot access '/content/runs/segment/': No such file or directory


In [47]:
import zipfile, os, shutil

shutil.rmtree('/content/mushroom_dataset', ignore_errors=True)
shutil.rmtree('/content/runs', ignore_errors=True)

zip_path = '/content/Mushrooms.yolov8.zip'
if not os.path.exists(zip_path):
    raise FileNotFoundError("Upload Mushrooms.yolov8.zip first!")

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/mushroom_dataset')

!cat /content/mushroom_dataset/data.yaml


train: ../train/images
val: ../valid/images
test: ../test/images

nc: 3
names: ['cap', 'stem', 'underside']

roboflow:
  workspace: ians-workspace-zjggb
  project: ians-workspace-zjggb
  version: dataset
  license: Private
  url: https://app.roboflow.com/ians-workspace-zjggb/ians-workspace-zjggb/dataset

In [48]:
!cat /content/mushroom_dataset/data.yaml
!echo "---"
!ls -la /content/mushroom_dataset/train/labels/ | head -5
!echo "---"
!find /content/ -maxdepth 1 -name "*.zip" -exec ls -lah {} \;


train: ../train/images
val: ../valid/images
test: ../test/images

nc: 3
names: ['cap', 'stem', 'underside']

roboflow:
  workspace: ians-workspace-zjggb
  project: ians-workspace-zjggb
  version: dataset
  license: Private
  url: https://app.roboflow.com/ians-workspace-zjggb/ians-workspace-zjggb/dataset---
total 1716
drwxr-xr-x 2 root root 36864 May 20 18:45 .
drwxr-xr-x 4 root root  4096 May 20 18:45 ..
-rw-r--r-- 1 root root     0 May 20 18:45 21_jpg.rf.skOpcSZ4aFmPfaBEQXYP.txt
-rw-r--r-- 1 root root     0 May 20 18:45 22_jpg.rf.4gO1r4Q0haUeMZkWU1kh.txt
---
-rw-r--r-- 1 root root 233M May 20 18:34 /content/Mushrooms.yolov8.zip


In [49]:
!find /content/ -name "best.pt" -type f 2>/dev/null
!ls -la /content/runs/segment/ 2>/dev/null || echo "No segment runs"


No segment runs


In [50]:
#train the model
from ultralytics import YOLO
model = YOLO('yolov8n-seg.pt')
model.train(data='/content/mushroom_dataset/data.yaml', epochs=100, imgsz=640, batch=16, patience=20)


Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/mushroom_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

RuntimeError: Dataset '/content/mushroom_dataset/data.yaml' error ❌ Dataset '/content/mushroom_dataset/data.yaml' images not found, missing path '/content/mushroom_dataset/valid/images'
Note dataset download directory is '/content/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'

In [ ]:
from google.colab import files
import os

weights_dir = '/content/runs/segment/mushroom_seg/weights'

for fname in ['best.pt', 'last.pt']:
    path = os.path.join(weights_dir, fname)
    if os.path.exists(path):
        print(f"Downloading {fname} ({os.path.getsize(path) / 1024 / 1024:.1f} MB)...")
        files.download(path)
    else:
        print(f"WARNING: {fname} not found at {path}")

print("\nDone. Check your browser's downloads folder.")


In [ ]:
from google.colab import files
files.download('/content/runs/segment/train-2/weights/best.pt')


In [ ]:
# Find ALL best.pt files anywhere under /content/
!find /content/ -name "best.pt" -type f 2>/dev/null | xargs ls -lah

# Also check the runs directory structure
!echo "---"
!ls -la /content/runs/segment/ 2>/dev/null || echo "No /content/runs/segment/"
!echo "---"
!ls -la /content/ultralytics/runs/segment/ 2>/dev/null || echo "No /content/ultralytics/runs/segment/"
from google.colab import files
# Replace with the actual path from the find command above
files.download('/content/runs/segment/train-3/weights/best.pt')
